In [1]:
# =============================================================================
# CS-781 Phase 5 -- Legal Statute Identification (LSI, MULTI-LABEL) with Arch #1 (Phase-1 model)
#
# Design
#   * Documents are cut into 1023-token chunks (+<s>); DOC_MODE / MAX_CHUNKS choose which chunks are used
#     ("spread" = chunks evenly spaced over the whole document, because a statute can be cited anywhere).
#   * Every chunk goes through the backbone (token states averaged into one chunk vector).
#   * POOL (chunk -> document):
#         attn / mean : pool the chunk vectors, then one classifier           (document-level head)
#         maxlogit    : classify EACH chunk, take the max logit over chunks    ("a statute is present if ANY chunk says so")
#         lse         : classify each chunk, log-mean-exp over chunks (soft max)
#   * Head: one logit per statute id (sigmoid / binary relevance). Bias starts at each label's training prior.
#   * Loss: BCE-with-logits, optional sqrt pos_weight for rare labels. KD uses soft-BCE against teacher sigmoids.
#   * Decision: a global threshold tuned on dev (cell [9]); at least one statute is always predicted.
#     Per-label thresholds + ensembles are handled on CPU in cell [12] with a 2-fold honesty check.
#
# Techniques (TECHNIQUES): full | lorma | lora | adapter | partial | normtune | kd
# Kaggle: paste each "# %%" block as its own cell, in order. FIRST RUN: keep TECHNIQUES=["lorma"], read the statistics
# printed by cell [4] and the "est. epoch time" line, then scale up.
# NOT yet run on Kaggle -- only the data/label/metric/threshold logic was tested (on fake data).
# =============================================================================

# %% [1] Imports & HF login
import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
import json, math, random, time, zipfile, pickle, gc, traceback, functools
from collections import Counter

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import f1_score, precision_recall_fscore_support

from kaggle_secrets import UserSecretsClient
from huggingface_hub import login, hf_hub_download

HF_TOKEN = UserSecretsClient().get_secret("HF")
os.environ["HF_TOKEN"] = HF_TOKEN
login(token=HF_TOKEN)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


Device: cuda


In [2]:
# %% [2] Config
TECHNIQUES = ["kd"]        # e.g. ["lorma"], ["adapter"], ["full"], ["kd"]   (run each in its own notebook if you like)
DOC_MODES = ["spread"]        # tail | head | headtail | tail4 | spread  (see cell [5]); every technique runs once per mode
FINAL_ALL = True             # True = train on train+dev for the fixed epochs (no dev score; threshold = THR_FINAL)
SEED = 48
SKIP_DONE = True
TEACHER_TECH = "full"         # KD only: which saved run is the teacher ("full" is the strongest; "lorma" also works)
TEACHER_CKPT = "/kaggle/input/datasets/akashmalgav2/phase5-full-lsi-best/lsi_best_full.pt"             # KD only: path to that run's lsi_best.pt (upload as a Kaggle dataset). "" = look in working dir, else retrain

ARTIFACT_PATH = "/kaggle/input/datasets/akashmalgav2/arch1final"                       # tokenizer (legal_bpe.json)
MODEL_CKPT_PATH = "/kaggle/input/datasets/akashmalgav2/phase2-model1/model_best.pt"       # Arch #1 Phase-1 checkpoint

HF_DATASET_REPO = "Exploration-Lab/CS-781-Capstone"
FILES = {"train": "Phase_5_LSI/Train/train.json", "dev": "Phase_5_LSI/Train/dev.json",
         "test": "Phase_5_LSI/Test/test.json"}

CHUNK_TOK = 1023
MAX_CHUNKS = 4                # chunks per document for tail4 / spread
MAX_DOC_WORDS = 16000         # very long documents: first HEAD_WORDS words + the tail
HEAD_WORDS = 4000
MAX_TRAIN_DOCS = None         # e.g. 6000 to cap training time; None = all
EVAL_CAP = 800                # dev docs used for the per-epoch model-selection score
POOL = "maxlogit"             # chunk -> document: attn | mean | maxlogit | lse
POOL_TOK = "lwan"             # token -> chunk: "mean" (average of token states) | "lwan" (LABEL-WISE ATTENTION: every statute id has its own
                              #   attention over the tokens, so a statute can be detected from the few tokens that mention it; needs POOL maxlogit|lse)
TEACHER_POOL_TOK = "mean"     # KD only: token pooling of the teacher checkpoint (the student may differ)
TEACHER_KEEP = None           # KD only: layers kept by the TEACHER checkpoint (None = all 12; a previous KD student used STUDENT_KEEP)
LOSS_POS_WEIGHT = "sqrt"      # none | sqrt   (pos_weight = sqrt(neg/pos), capped)
POS_WEIGHT_CAP = 20.0

CHUNKS_PER_BATCH = 16         # max chunks per forward pass (docs are short, batches are length-bucketed; ~9 GB for full fine-tuning)
TARGET_DOC_BATCH = 16
WEIGHT_DECAY, WARMUP_FRAC, DROPOUT_FT = 0.01, 0.06, 0.1

THR_GRID = np.round(np.arange(0.05, 0.91, 0.05), 2)
THR_FINAL = 0.30              # only used when FINAL_ALL=True (no dev set to tune on)
FORCE_TOP1 = True             # always predict at least one statute per document (compare True/False on dev)
MAX_PRED_ID = None            # the official scorer rejects any predicted id >= (max gold id in the test reference + 1).
                              # If Codabench ever says "Invalid/duplicate label in prediction", set this to a lower id.

LORMA_R, LORMA_ALPHA = 8, 8
ADAPTER_DIM = 32
PARTIAL_LAYERS = 4
STUDENT_KEEP = [0, 1, 3, 4, 6, 7, 9, 10, 11]   # KD student keeps 9 of 12 Phase-1 layers
KD_T, KD_ALPHA = 2.0, 0.5

HP = {   # starting points -- adjust after the epoch-time estimate and the first dev scores
    "full":     dict(epochs=3, lr_body=3e-5, lr_head=1e-3),
    "lorma":    dict(epochs=6, lr_body=4e-4, lr_head=1e-3),
    "lora":     dict(epochs=6, lr_body=4e-4, lr_head=1e-3),
    "adapter":  dict(epochs=4, lr_body=4e-4, lr_head=1e-3),
    "partial":  dict(epochs=5, lr_body=1e-4, lr_head=1e-3),
    "normtune": dict(epochs=8, lr_body=1e-3, lr_head=1e-3),
    "kd":       dict(epochs=5, lr_body=1e-4, lr_head=1e-3),
}
for _t in TECHNIQUES:
    assert _t in HP, _t
if "kd" in TECHNIQUES and TEACHER_TECH in TECHNIQUES:
    assert TECHNIQUES.index(TEACHER_TECH) < TECHNIQUES.index("kd"), "put the teacher technique before 'kd'"

ROOT_OUT = "/kaggle/working"
DOC_MODE = DOC_MODES[0]
def out_dir_for(tech):
    return f"{ROOT_OUT}/cs781_phase5_arch1_{tech}_{DOC_MODE}_{POOL}" + ("" if POOL_TOK == "mean" else "_" + POOL_TOK) + f"_s{SEED}" + ("_all" if FINAL_ALL else "")
random.seed(SEED); torch.manual_seed(SEED); np.random.seed(SEED)
print("Will run:", TECHNIQUES, "| doc modes:", DOC_MODES, "| pool:", POOL, "| seed:", SEED, "| final-all:", FINAL_ALL)


Will run: ['kd'] | doc modes: ['spread'] | pool: maxlogit | seed: 48 | final-all: True


In [3]:




# %% [3] Tokenizer (Phase-1 BPE), cached; documents are stored as uint16 arrays to save RAM
with open(f"{ARTIFACT_PATH}/legal_bpe.json", "r", encoding="utf-8") as f:
    _tok = json.load(f)
vocab = _tok["itos"]
merges = [tuple(p) for p in _tok["merges"]]
token_to_id = {tok: i for i, tok in enumerate(vocab)}
merges_rank = {p: i for i, p in enumerate(merges)}
PAD_ID, UNK_ID = token_to_id["<pad>"], token_to_id["<unk>"]
BOS_ID, EOS_ID = token_to_id["<s>"], token_to_id["</s>"]
VOCAB_SIZE = len(vocab)
assert VOCAB_SIZE < 65536
EOW = "\u0100"

def word_to_symbols(word):
    return tuple(chr(b) for b in word.encode("utf-8", "ignore")) + (EOW,)

def bpe_encode_word(word):
    symbols = list(word_to_symbols(word))
    while len(symbols) > 1:
        pairs = [(symbols[i], symbols[i + 1]) for i in range(len(symbols) - 1)]
        ranked = [(merges_rank[p], p) for p in pairs if p in merges_rank]
        if not ranked:
            break
        _, best = min(ranked)
        new_symbols, i = [], 0
        while i < len(symbols):
            if i < len(symbols) - 1 and symbols[i] == best[0] and symbols[i + 1] == best[1]:
                new_symbols.append(best[0] + best[1]); i += 2
            else:
                new_symbols.append(symbols[i]); i += 1
        symbols = new_symbols
    return symbols

@functools.lru_cache(maxsize=None)
def encode_word_cached(word):
    return tuple(token_to_id.get(sym, UNK_ID) for sym in bpe_encode_word(word))

def encode_doc(text):
    words = str(text).split()
    if len(words) > MAX_DOC_WORDS:
        words = words[:HEAD_WORDS] + words[-(MAX_DOC_WORDS - HEAD_WORDS):]
    ids = []
    for w in words:
        ids.extend(encode_word_cached(w))
    return np.asarray(ids, dtype=np.uint16)


# %% [4] Load Phase-5 data, build the label space, print statistics
def hf_local(path):
    return hf_hub_download(repo_id=HF_DATASET_REPO, filename=path, repo_type="dataset", token=HF_TOKEN)

def read_records(path):
    raw = json.load(open(hf_local(path), encoding="utf-8"))
    if isinstance(raw, dict) and "data" in raw and isinstance(raw["data"], list):
        raw = raw["data"]
    recs = []
    if isinstance(raw, list):
        for i, r in enumerate(raw):
            assert isinstance(r, dict), f"{path}: record {i} is {type(r)}"
            recs.append(r)
    elif isinstance(raw, dict):
        for k, v in raw.items():
            if isinstance(v, str):
                recs.append({"id": k, "text": v})
            else:
                d = dict(v); d.setdefault("id", k); recs.append(d)
    else:
        raise ValueError(f"{path}: unexpected top-level type {type(raw)}")
    return recs

def parse_labels(x):
    if x is None:
        return []
    if isinstance(x, (int, float, str)):
        x = [x]
    return sorted({int(v) for v in x})

def load_split(name, cap=None):
    recs = read_records(FILES[name])
    print(f"{name}: {len(recs)} records | keys of first record: {list(recs[0].keys())}")
    if name == "test" and any(r.get("labels") not in (None, [], "") for r in recs):
        print("  !! WARNING: the test file contains a populated 'labels' field. It is IGNORED here and must never be used.")
    if cap is not None and len(recs) > cap:
        rng = random.Random(SEED); recs = rng.sample(recs, cap)
        print(f"  -> random subset of {cap}")
    docs, t0 = [], time.time()
    for n, r in enumerate(recs):
        labs = [] if name == "test" else parse_labels(r.get("labels"))
        docs.append({"id": str(r["id"]), "toks": encode_doc(r.get("text") or ""), "labels": labs})
        if (n + 1) % 2000 == 0:
            print(f"  tokenised {n+1}/{len(recs)} docs ({time.time()-t0:.0f}s)")
    if name != "test":
        seen, uniq = set(), []
        for d in docs:
            key = (d["id"], tuple(d["labels"]), hash(d["toks"].tobytes()))
            if key not in seen:
                seen.add(key); uniq.append(d)
        print(f"  {name}: removed {len(docs)-len(uniq)} exact duplicates | "
              f"{len(uniq)-len({d['id'] for d in uniq})} further docs share an id with different content (kept)")
        docs = uniq
    else:
        assert len({d["id"] for d in docs}) == len(docs), "duplicate ids in TEST (need one prediction per id)"
    for i, d in enumerate(docs):
        d["uid"] = f"{d['id']}#{i}"
    print(f"  {name} ready in {time.time()-t0:.0f}s")
    return docs

train_docs = load_split("train", MAX_TRAIN_DOCS)
dev_docs = load_split("dev")
test_docs = load_split("test")

# label space = every statute id seen in train or dev (the head has one logit per id)
all_labels = sorted({l for d in train_docs + dev_docs for l in d["labels"]})
lab2idx = {l: i for i, l in enumerate(all_labels)}
N_LAB = len(all_labels)
for d in train_docs + dev_docs:
    d["y"] = [lab2idx[l] for l in d["labels"]]
for d in test_docs:
    d["y"] = []
assert all(len(d["y"]) > 0 for d in train_docs), "some training docs have no labels"
print(f"label space: {N_LAB} statute ids (min id {all_labels[0]}, max id {all_labels[-1]})")

def stats(name, docs):
    n = np.array([len(d["toks"]) for d in docs])
    print(f"{name}: {len(docs)} docs | tokens median {int(np.median(n))} p90 {int(np.percentile(n, 90))} max {int(n.max())} "
          f"| chunks(1023) median {int(np.median(np.ceil(n/1023)))} p90 {int(np.percentile(np.ceil(n/1023), 90))}")
    if name != "test":
        k = np.array([len(d["y"]) for d in docs])
        print(f"   labels per doc: mean {k.mean():.2f} median {int(np.median(k))} max {int(k.max())}")
for nm, ds in [("train", train_docs), ("dev", dev_docs), ("test", test_docs)]:
    stats(nm, ds)

pos_cnt = np.zeros(N_LAB)
for d in train_docs:
    for j in d["y"]:
        pos_cnt[j] += 1
print("train positives per label: median", int(np.median(pos_cnt)), "| max", int(pos_cnt.max()),
      "| labels with <5:", int((pos_cnt < 5).sum()), "| <10:", int((pos_cnt < 10).sum()), "| <20:", int((pos_cnt < 20).sum()))
top = np.argsort(-pos_cnt)[:10]
print("most frequent statute ids (id: train count):", {all_labels[j]: int(pos_cnt[j]) for j in top})
dev_pos = np.zeros(N_LAB)
for d in dev_docs:
    for j in d["y"]:
        dev_pos[j] += 1
print(f"dev: {int((dev_pos > 0).sum())} labels have >=1 dev positive | {int(((dev_pos > 0) & (pos_cnt == 0)).sum())} of them never appear in train")

rng_e = random.Random(SEED)
eval_subset = dev_docs if len(dev_docs) <= EVAL_CAP else rng_e.sample(dev_docs, EVAL_CAP)
fit_docs = train_docs + dev_docs if FINAL_ALL else train_docs
eval_docs_epoch = [] if FINAL_ALL else eval_subset
print(f"fit docs {len(fit_docs)} | per-epoch eval docs {len(eval_docs_epoch)} | final report docs {0 if FINAL_ALL else len(dev_docs)}")

fit_cnt = np.zeros(N_LAB)
for d in fit_docs:
    for j in d["y"]:
        fit_cnt[j] += 1
Nf = len(fit_docs)
prior = (fit_cnt + 1.0) / (Nf + 2.0)
prior_logit = torch.tensor(np.log(prior / (1 - prior)), dtype=torch.float32)
if LOSS_POS_WEIGHT == "none":
    pw = np.ones(N_LAB)
else:
    pw = np.clip(np.sqrt((Nf - fit_cnt) / np.maximum(fit_cnt, 1)), 1.0, POS_WEIGHT_CAP)
    pw[fit_cnt == 0] = 1.0
pos_w = torch.tensor(pw, dtype=torch.float32).to(device)
print("pos_weight: min %.2f median %.2f max %.2f" % (pw.min(), np.median(pw), pw.max()))


# %% [5] Chunking, chunk selection, datasets
def chunk_starts(n, C):
    """Start offsets of 1023-token chunks; the last chunk is tail-aligned (may overlap the previous one)."""
    if n <= C:
        return [0]
    st = list(range(0, n - C + 1, C))
    if st[-1] != n - C:
        st.append(n - C)
    return st

def select_starts(starts):
    m = len(starts)
    if DOC_MODE == "tail":
        idx = [m - 1]
    elif DOC_MODE == "head":
        idx = [0]
    elif DOC_MODE == "headtail":
        idx = sorted({0, m - 1})
    elif DOC_MODE == "tail4":
        idx = list(range(max(0, m - MAX_CHUNKS), m))
    elif DOC_MODE == "spread":
        idx = sorted({int(round(x)) for x in np.linspace(0, m - 1, min(m, MAX_CHUNKS))})
    else:
        raise ValueError(DOC_MODE)
    return [starts[i] for i in idx]

def chunks_per_doc():
    return {"tail": 1, "head": 1, "headtail": 2}.get(DOC_MODE, MAX_CHUNKS)

class DocDataset(Dataset):
    def __init__(self, docs):
        self.docs = docs
    def __len__(self):
        return len(self.docs)
    def __getitem__(self, i):
        d = self.docs[i]
        toks = d["toks"]
        starts = select_starts(chunk_starts(len(toks), CHUNK_TOK))
        chunks = [np.concatenate(([BOS_ID], toks[s:s + CHUNK_TOK])).astype(np.int64) for s in starts]
        y = np.zeros(N_LAB, dtype=np.float32)
        y[d["y"]] = 1.0
        return {"chunks": chunks, "y": y}

def collate(batch):
    chunks = [c for x in batch for c in x["chunks"]]
    Nc, T = len(chunks), max(len(c) for c in chunks)
    ids = torch.full((Nc, T), PAD_ID, dtype=torch.long)
    lens = torch.zeros(Nc, dtype=torch.long)
    for i, c in enumerate(chunks):
        ids[i, :len(c)] = torch.from_numpy(c); lens[i] = len(c)
    B, Cm = len(batch), max(len(x["chunks"]) for x in batch)
    cidx = torch.zeros((B, Cm), dtype=torch.long)
    cmask = torch.zeros((B, Cm), dtype=torch.bool)
    k = 0
    for b, x in enumerate(batch):
        for j in range(len(x["chunks"])):
            cidx[b, j] = k; cmask[b, j] = True; k += 1
    lab = torch.from_numpy(np.stack([x["y"] for x in batch]))
    return ids, lens, cidx, cmask, lab

def loader_settings():
    dpb = max(1, CHUNKS_PER_BATCH // chunks_per_doc())
    accum = max(1, TARGET_DOC_BATCH // dpb)
    return dpb, accum

class LengthBucketSampler:
    """Shuffled batches of similar-length documents (less padding => faster). Evaluation keeps the plain sequential order."""
    def __init__(self, docs, batch_size, mega=50, seed=0):
        self.key = [min(len(d["toks"]), CHUNK_TOK) for d in docs]     # padded length proxy
        self.bs, self.mega, self.seed, self.epoch = batch_size, mega, seed, 0
    def __iter__(self):
        self.epoch += 1
        rng = random.Random(self.seed * 1000 + self.epoch)
        idx = list(range(len(self.key))); rng.shuffle(idx)
        M, batches = self.bs * self.mega, []
        for i in range(0, len(idx), M):
            blk = sorted(idx[i:i + M], key=lambda j: self.key[j])
            batches += [blk[k:k + self.bs] for k in range(0, len(blk), self.bs)]
        rng.shuffle(batches)
        return iter(batches)
    def __len__(self):
        n, M = len(self.key), self.bs * self.mega
        return (n // M) * self.mega + math.ceil((n % M) / self.bs)

def make_loader(docs, shuffle):
    dpb, _ = loader_settings()
    ds = DocDataset(docs)
    if shuffle:
        return DataLoader(ds, batch_sampler=LengthBucketSampler(docs, dpb, seed=SEED), collate_fn=collate, num_workers=0)
    return DataLoader(ds, batch_size=dpb, shuffle=False, collate_fn=collate, num_workers=0)

# %% [6] Model: Arch #1 backbone + PEFT modules + multi-label document head
class RMSNorm(nn.Module):
    def __init__(self, dim, eps=1e-5):
        super().__init__()
        self.eps = eps
        self.weight = nn.Parameter(torch.ones(dim))
    def forward(self, x):
        return x * torch.rsqrt(x.pow(2).mean(-1, keepdim=True) + self.eps) * self.weight

def build_rope_cache(seq_len, head_dim, device, base=10000.0):
    inv_freq = 1.0 / (base ** (torch.arange(0, head_dim, 2, device=device).float() / head_dim))
    t = torch.arange(seq_len, device=device).float()
    freqs = torch.einsum("i,j->ij", t, inv_freq)
    emb = torch.cat((freqs, freqs), dim=-1)
    return emb.cos()[None, None, :, :], emb.sin()[None, None, :, :]

def rotate_half(x):
    x1, x2 = x.chunk(2, dim=-1)
    return torch.cat((-x2, x1), dim=-1)

def apply_rope(q, k, cos, sin):
    return (q * cos) + (rotate_half(q) * sin), (k * cos) + (rotate_half(k) * sin)

class LoRMAPlus(nn.Module):
    """h' = h + (alpha/r) B(A h) = ((alpha/r) B A + I) W0 x with h = W0 x (multiplicative; B = 0 at init)."""
    def __init__(self, d, r, alpha):
        super().__init__()
        self.A = nn.Parameter(torch.empty(r, d))
        nn.init.kaiming_uniform_(self.A, a=math.sqrt(5))
        self.B = nn.Parameter(torch.zeros(d, r))
        self.scale = alpha / r
    def forward(self, h):
        return h + self.scale * F.linear(F.linear(h, self.A), self.B)

class LoRAAdapter(nn.Module):
    """Additive update (alpha/r) B(A x), added to W0 x (LoRA; B = 0 at init)."""
    def __init__(self, d, r, alpha):
        super().__init__()
        self.A = nn.Parameter(torch.empty(r, d))
        nn.init.kaiming_uniform_(self.A, a=math.sqrt(5))
        self.B = nn.Parameter(torch.zeros(d, r))
        self.scale = alpha / r
    def forward(self, x):
        return self.scale * F.linear(F.linear(x, self.A), self.B)

class BottleneckAdapter(nn.Module):
    """Houlsby-style adapter: h + up(gelu(down(h))); `up` starts at zero (identity at init)."""
    def __init__(self, d, m):
        super().__init__()
        self.down = nn.Linear(d, m)
        self.up = nn.Linear(m, d)
        nn.init.normal_(self.down.weight, std=0.02); nn.init.zeros_(self.down.bias)
        nn.init.zeros_(self.up.weight); nn.init.zeros_(self.up.bias)
    def forward(self, h):
        return h + self.up(F.gelu(self.down(h)))

class CausalSelfAttention(nn.Module):
    def __init__(self, n_embd, n_head, dropout):
        super().__init__()
        self.n_head, self.head_dim, self.n_embd = n_head, n_embd // n_head, n_embd
        self.qkv = nn.Linear(n_embd, 3 * n_embd, bias=False)
        self.proj = nn.Linear(n_embd, n_embd, bias=False)
        self.dropout = dropout
        self.adapt_mode = None
        self.q_ad = None
        self.v_ad = None
    def forward(self, x, cos, sin):
        B, T, C = x.shape
        q, k, v = self.qkv(x).split(C, dim=-1)
        if self.adapt_mode == "lorma":
            q, v = self.q_ad(q), self.v_ad(v)
        elif self.adapt_mode == "lora":
            q, v = q + self.q_ad(x), v + self.v_ad(x)
        q = q.view(B, T, self.n_head, self.head_dim).transpose(1, 2)
        k = k.view(B, T, self.n_head, self.head_dim).transpose(1, 2)
        v = v.view(B, T, self.n_head, self.head_dim).transpose(1, 2)
        q, k = apply_rope(q, k, cos[:, :, :T, :], sin[:, :, :T, :])
        out = F.scaled_dot_product_attention(q, k, v, is_causal=True,
                                             dropout_p=self.dropout if self.training else 0.0)
        return self.proj(out.transpose(1, 2).contiguous().view(B, T, C))

class SwiGLU(nn.Module):
    def __init__(self, n_embd, mult, dropout):
        super().__init__()
        hidden = ((int(mult * n_embd) + 63) // 64) * 64
        self.w1 = nn.Linear(n_embd, hidden, bias=False)
        self.w2 = nn.Linear(n_embd, hidden, bias=False)
        self.w3 = nn.Linear(hidden, n_embd, bias=False)
        self.drop = nn.Dropout(dropout)
    def forward(self, x):
        return self.drop(self.w3(F.silu(self.w1(x)) * self.w2(x)))

class Block(nn.Module):
    def __init__(self, n_embd, n_head, ffn_mult, dropout):
        super().__init__()
        self.ln1 = RMSNorm(n_embd)
        self.attn = CausalSelfAttention(n_embd, n_head, dropout)
        self.ln2 = RMSNorm(n_embd)
        self.ffn = SwiGLU(n_embd, ffn_mult, dropout)
        self.ad1 = None
        self.ad2 = None
    def forward(self, x, cos, sin):
        a = self.attn(self.ln1(x), cos, sin)
        if self.ad1 is not None:
            a = self.ad1(a)
        x = x + a
        f = self.ffn(self.ln2(x))
        if self.ad2 is not None:
            f = self.ad2(f)
        return x + f

class LegalLLaMA(nn.Module):
    def __init__(self, vocab_size, n_layer, n_head, n_embd, block_size, ffn_mult, dropout):
        super().__init__()
        self.block_size, self.head_dim = block_size, n_embd // n_head
        self.tok_emb = nn.Embedding(vocab_size, n_embd)
        self.drop = nn.Dropout(dropout)
        self.blocks = nn.ModuleList([Block(n_embd, n_head, ffn_mult, dropout) for _ in range(n_layer)])
        self.ln_f = RMSNorm(n_embd)
        self.head = nn.Linear(n_embd, vocab_size, bias=False)
        self.tok_emb.weight = self.head.weight
        cos, sin = build_rope_cache(block_size, self.head_dim, device="cpu")
        self.register_buffer("rope_cos", cos, persistent=False)
        self.register_buffer("rope_sin", sin, persistent=False)
    def forward_hidden(self, idx):
        x = self.drop(self.tok_emb(idx))
        cos, sin = self.rope_cos.to(idx.device), self.rope_sin.to(idx.device)
        for blk in self.blocks:
            x = blk(x, cos, sin)
        return self.ln_f(x)


class DocModel(nn.Module):
    """Backbone over every selected chunk -> chunk-level scores -> multi-label document logits.
       POOL_TOK="mean": mean of token states -> chunk vector -> (see POOL).
       POOL_TOK="lwan": label-wise attention: logit_l = o_l . sum_t softmax_t(q_l . h_t) h_t + b_l, then max/lse over chunks."""
    def __init__(self, backbone, d, n_lab, dropout, pool, pool_tok="mean"):
        super().__init__()
        self.backbone, self.pool, self.pool_tok = backbone, pool, pool_tok
        self.drop = nn.Dropout(dropout)
        self.classifier = nn.Linear(d, n_lab)
        nn.init.normal_(self.classifier.weight, std=0.02); nn.init.zeros_(self.classifier.bias)
        self.attn_score = nn.Linear(d, 1)
        nn.init.zeros_(self.attn_score.weight); nn.init.zeros_(self.attn_score.bias)   # uniform weights at init
        if pool_tok == "lwan":
            assert pool in ("maxlogit", "lse"), "lwan needs POOL = maxlogit or lse"
            self.lw_q = nn.Parameter(torch.randn(n_lab, d) * 0.05)     # one attention query per statute id
            self.lw_o = nn.Parameter(torch.randn(n_lab, d) * 0.02)     # one output vector per statute id
            self.lw_b = nn.Parameter(torch.zeros(n_lab))
    def head_parameters(self):
        ps = list(self.classifier.parameters()) + list(self.attn_score.parameters())
        if self.pool_tok == "lwan":
            ps += [self.lw_q, self.lw_o, self.lw_b]
        return ps
    def init_bias(self, b):
        with torch.no_grad():
            (self.lw_b if self.pool_tok == "lwan" else self.classifier.bias).copy_(b)
    def forward(self, ids, lens, cidx, cmask):
        h = self.backbone.forward_hidden(ids)                                   # (Nc, T, d)
        T = h.size(1)
        valid = torch.arange(T, device=ids.device)[None, :] < lens[:, None]    # (Nc, T)
        if self.pool_tok == "lwan":
            hf = self.drop(h.float())
            sc = torch.einsum("ntd,ld->nlt", hf, self.lw_q).float().masked_fill(~valid.unsqueeze(1), -1e4)   # (Nc, L, T)
            al = torch.softmax(sc, dim=-1)
            ctx = torch.einsum("nlt,ntd->nld", al, hf).float()                  # (Nc, L, d)
            cl = (ctx * self.lw_o.unsqueeze(0)).sum(-1) + self.lw_b             # (Nc, L) chunk-level logits
            logits = cl[cidx].masked_fill(~cmask.unsqueeze(-1), -1e4)           # (B, Cmax, L)
            if self.pool == "maxlogit":
                return logits.max(1).values
            return torch.logsumexp(logits, dim=1) - torch.log(cmask.sum(1, keepdim=True).float())
        tm = valid.unsqueeze(-1).float()
        cv = (h.float() * tm).sum(1) / lens.unsqueeze(-1).float()               # (Nc, d)
        D = self.drop(cv[cidx])                                                 # (B, Cmax, d)
        if self.pool in ("attn", "mean"):
            if self.pool == "attn":
                s_ = self.attn_score(D).squeeze(-1).masked_fill(~cmask, -1e4)
                w = torch.softmax(s_, dim=1)
            else:
                w = cmask.float() / cmask.float().sum(1, keepdim=True)
            return self.classifier((w.unsqueeze(-1) * D).sum(1))                # (B, L)
        logits = self.classifier(D).float().masked_fill(~cmask.unsqueeze(-1), -1e4)   # (B, Cmax, L)
        if self.pool == "maxlogit":
            return logits.max(1).values
        if self.pool == "lse":                                                  # log-mean-exp over chunks
            return torch.logsumexp(logits, dim=1) - torch.log(cmask.sum(1, keepdim=True).float())
        raise ValueError(self.pool)

ckpt = torch.load(MODEL_CKPT_PATH, map_location="cpu")
cfg = ckpt["config"]
assert cfg["vocab_size"] == VOCAB_SIZE, (cfg["vocab_size"], VOCAB_SIZE)
assert CHUNK_TOK + 1 <= cfg["block_size"], f"CHUNK_TOK+1 exceeds the model context {cfg['block_size']}"
print(f"Phase-1 model: {cfg['n_layer']} layers, d={cfg['n_embd']}, context {cfg['block_size']}")

def make_student_state(sd, keep):
    new = {}
    for k, v in sd.items():
        if k.startswith("blocks."):
            parts = k.split(".")
            old = int(parts[1])
            if old in keep:
                parts[1] = str(keep.index(old))
                new[".".join(parts)] = v
        else:
            new[k] = v
    return new

def build_model(technique, keep_layers=None, pool_tok=None):
    sd, n_layer = ckpt["model_state_dict"], cfg["n_layer"]
    if keep_layers is not None:
        sd, n_layer = make_student_state(sd, keep_layers), len(keep_layers)
    bb = LegalLLaMA(cfg["vocab_size"], n_layer, cfg["n_head"], cfg["n_embd"],
                    cfg["block_size"], cfg["ffn_mult"], DROPOUT_FT)
    bb.load_state_dict(sd)
    d = cfg["n_embd"]
    if technique in ("lorma", "lora", "adapter", "partial", "normtune"):
        for p in bb.parameters():
            p.requires_grad = False
        if technique in ("lorma", "lora"):
            for blk in bb.blocks:
                blk.attn.adapt_mode = technique
                make = LoRMAPlus if technique == "lorma" else LoRAAdapter
                blk.attn.q_ad, blk.attn.v_ad = make(d, LORMA_R, LORMA_ALPHA), make(d, LORMA_R, LORMA_ALPHA)
        elif technique == "adapter":
            for blk in bb.blocks:
                blk.ad1, blk.ad2 = BottleneckAdapter(d, ADAPTER_DIM), BottleneckAdapter(d, ADAPTER_DIM)
        elif technique == "partial":
            for blk in bb.blocks[-PARTIAL_LAYERS:]:
                for p in blk.parameters():
                    p.requires_grad = True
            for p in bb.ln_f.parameters():
                p.requires_grad = True
        elif technique == "normtune":
            for m in bb.modules():
                if isinstance(m, RMSNorm):
                    m.weight.requires_grad = True
    net = DocModel(bb, d, N_LAB, DROPOUT_FT, POOL, pool_tok or POOL_TOK).to(device)
    net.init_bias(prior_logit.to(device))                      # start every label at its training prior
    n_tot = sum(p.numel() for p in net.parameters())
    n_trn = sum(p.numel() for p in net.parameters() if p.requires_grad)
    print(f"[{technique}{' student' if keep_layers else ''}] trainable {n_trn/1e6:.3f}M of {n_tot/1e6:.1f}M params ({100*n_trn/n_tot:.2f}%)")
    return net


# %% [7] Inference and multi-label metrics (numpy; macro-F1 over labels that have >= 1 positive in the evaluated set)
@torch.no_grad()
def predict_probs(net, docs):
    net.eval()
    out, k = {}, 0
    for ids, lens, cidx, cmask, _ in make_loader(docs, False):
        with torch.autocast(device_type="cuda", dtype=torch.float16, enabled=torch.cuda.is_available()):
            logits = net(ids.to(device), lens.to(device), cidx.to(device), cmask.to(device))
        probs = torch.sigmoid(logits.float()).cpu().numpy()
        for j in range(len(probs)):
            out[docs[k]["uid"]] = probs[j].astype(np.float32); k += 1
    return out

def label_matrix(docs):
    Y = np.zeros((len(docs), N_LAB), dtype=bool)
    for i, d in enumerate(docs):
        Y[i, d["y"]] = True
    return Y

def prob_matrix(probs_by_uid, docs):
    return np.stack([probs_by_uid[d["uid"]] for d in docs])

def apply_thr(P, thr, top1=None):
    """P: (N, L) probabilities, thr: scalar or (L,). If FORCE_TOP1, at least one label per document ('one or more')."""
    pred = P >= thr
    if FORCE_TOP1 if top1 is None else top1:
        empty = ~pred.any(1)
        if empty.any():
            pred[np.where(empty)[0], P[empty].argmax(1)] = True
    return pred

def f1_per_label(Y, pred):
    tp = (pred & Y).sum(0).astype(float); fp = (pred & ~Y).sum(0); fn = (~pred & Y).sum(0)
    den = 2 * tp + fp + fn
    return np.where(den > 0, 2 * tp / np.maximum(den, 1), 0.0)

def macro_f1(Y, pred):
    sup = Y.any(0)
    return float(f1_per_label(Y, pred)[sup].mean())

def micro_f1(Y, pred):
    tp = float((pred & Y).sum()); fp = float((pred & ~Y).sum()); fn = float((~pred & Y).sum())
    return 2 * tp / max(2 * tp + fp + fn, 1)

def subset_acc(Y, pred):
    return float((pred == Y).all(1).mean())

def best_global_thr(Y, P, grid=None):
    grid = THR_GRID if grid is None else grid
    res = [(macro_f1(Y, apply_thr(P, t)), float(t)) for t in grid]
    return max(res)                                  # (macro-F1, threshold)

def report(probs_by_uid, docs):
    Y, P = label_matrix(docs), prob_matrix(probs_by_uid, docs)
    f1, thr = best_global_thr(Y, P)
    pred = apply_thr(P, thr)
    per = f1_per_label(Y, pred)
    sup = Y.any(0)
    n_cols = max(all_labels[j] for j in np.where(sup)[0]) + 1          # official scorer: columns 0..max gold id
    official = float(per[sup].sum() / n_cols)                           # absent ids count as F1 = 0 (zero_division=0)
    print(f"dev macro-F1 over labels present in dev {f1:.4f} (global thr {thr:.2f}; {int(sup.sum())} labels) | "
          f"OFFICIAL-STYLE macro-F1 (all {n_cols} id columns, absent ids = 0) {official:.4f} | "
          f"micro-F1 {micro_f1(Y, pred):.4f} | subset acc {subset_acc(Y, pred):.4f}")
    print(f"  avg #labels per doc: predicted {pred.sum(1).mean():.2f} vs gold {Y.sum(1).mean():.2f} | @thr 0.5 macro-F1 {macro_f1(Y, apply_thr(P, 0.5)):.4f}")
    for lo, hi in [(0, 100), (100, 500), (500, 2000), (2000, 10**9)]:
        m = sup & (pos_cnt >= lo) & (pos_cnt < hi)
        if m.any():
            print(f"  labels with {lo}-{hi-1 if hi < 10**9 else '+'} train positives: {int(m.sum()):4d} labels | macro-F1 {float(per[m].mean()):.4f}")
    return f1, thr


# %% [8] Training function (gradient accumulation; optional teacher for distillation)
def fit(model, loader, epochs, lr_body, lr_head, best_path, teacher=None):
    _, ACCUM = loader_settings()
    head_params = model.head_parameters()
    head_ids = {id(p) for p in head_params}
    body_params = [p for p in model.parameters() if p.requires_grad and id(p) not in head_ids]
    trainable = [p for p in model.parameters() if p.requires_grad]
    groups = [{"params": head_params, "lr": lr_head, "base": lr_head}]
    if body_params:
        groups.insert(0, {"params": body_params, "lr": lr_body, "base": lr_body})
    optimizer = torch.optim.AdamW(groups, weight_decay=WEIGHT_DECAY, betas=(0.9, 0.98))
    scaler = torch.amp.GradScaler("cuda", enabled=torch.cuda.is_available())
    steps_per_epoch = math.ceil(len(loader) / ACCUM)
    total_steps = epochs * steps_per_epoch
    warm = max(1, int(WARMUP_FRAC * total_steps))
    def lr_factor(step):
        if step < warm:
            return (step + 1) / warm
        prog = (step - warm) / max(1, total_steps - warm)
        return max(0.02, 0.5 * (1 + math.cos(math.pi * prog)))

    best_f1, step, t0, history, eta_done = -1.0, 0, time.time(), [], False
    if teacher is not None:
        teacher.eval()
    for epoch in range(1, epochs + 1):
        model.train()
        run_loss, n_b = 0.0, 0
        optimizer.zero_grad(set_to_none=True)
        for bi, (ids, lens, cidx, cmask, lab) in enumerate(loader):
            ids, lens, cidx, cmask, lab = ids.to(device), lens.to(device), cidx.to(device), cmask.to(device), lab.to(device)
            with torch.autocast(device_type="cuda", dtype=torch.float16, enabled=torch.cuda.is_available()):
                logits = model(ids, lens, cidx, cmask)
                if teacher is not None:
                    with torch.no_grad():
                        t_logits = teacher(ids, lens, cidx, cmask)
            logits = logits.float()
            loss = F.binary_cross_entropy_with_logits(logits, lab, pos_weight=pos_w, reduction="none").sum(1).mean()
            if teacher is not None:
                soft = torch.sigmoid(t_logits.float() / KD_T)
                kd = F.binary_cross_entropy_with_logits(logits / KD_T, soft, reduction="none").sum(1).mean()
                loss = (1 - KD_ALPHA) * loss + KD_ALPHA * kd
            scaler.scale(loss / ACCUM).backward()
            run_loss += loss.item(); n_b += 1
            if (bi + 1) % ACCUM == 0 or bi + 1 == len(loader):
                for g in optimizer.param_groups:
                    g["lr"] = g["base"] * lr_factor(step)
                scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(trainable, 1.0)
                scaler.step(optimizer); scaler.update()
                optimizer.zero_grad(set_to_none=True)
                step += 1
            if not eta_done and bi == 49:
                per = (time.time() - t0) / 50
                print(f"  est. epoch time {per*len(loader)/60:.1f} min ({len(loader)} batches, {per:.2f}s/batch) "
                      f"-> {epochs} epochs ~ {epochs*per*len(loader)/60:.0f} min (+ eval)")
                eta_done = True
        rec_h = {"epoch": epoch, "train_loss": run_loss / max(1, n_b)}
        if eval_docs_epoch:
            Ye = label_matrix(eval_docs_epoch); Pe = prob_matrix(predict_probs(model, eval_docs_epoch), eval_docs_epoch)
            f1, thr = best_global_thr(Ye, Pe)
            rec_h["dev_f1"], rec_h["thr"] = f1, thr
            print(f"[epoch {epoch}] loss {rec_h['train_loss']:.3f} | dev(sub) macro-F1 {f1:.4f} @thr {thr:.2f} "
                  f"(micro {micro_f1(Ye, apply_thr(Pe, thr)):.4f}) | {(time.time()-t0)/60:.1f}m")
            if f1 > best_f1:
                best_f1 = f1
                torch.save({"model": model.state_dict(), "epoch": epoch, "dev_f1": f1}, best_path)
                print("  -> new best checkpoint saved")
        else:
            print(f"[epoch {epoch}] loss {rec_h['train_loss']:.3f} (final run on train+dev; no dev score) | {(time.time()-t0)/60:.1f}m")
            torch.save({"model": model.state_dict(), "epoch": epoch, "dev_f1": None}, best_path)
        history.append(rec_h)
    model.load_state_dict(torch.load(best_path, map_location="cpu")["model"])
    return history


# %% [9] One technique end-to-end (train -> dev report -> test predictions -> save probabilities + efficiency numbers)
def set_seed(s):
    random.seed(s); torch.manual_seed(s); np.random.seed(s)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(s)

def make_submission(test_P, thr):
    pred = apply_thr(test_P, thr)
    allowed = np.array([MAX_PRED_ID is None or l <= MAX_PRED_ID for l in all_labels])
    out = []
    for d, row_p, row in zip(test_docs, test_P, pred):
        ids_ = sorted({int(all_labels[j]) for j in np.where(row & allowed)[0]})
        if not ids_ and FORCE_TOP1:
            ids_ = [int(all_labels[int(np.where(allowed, row_p, -1.0).argmax())])]
        assert FORCE_TOP1 is False or len(ids_) >= 1
        out.append({"id": d["id"], "pred": ids_})
    return out

def run_technique(tech):
    out_dir = out_dir_for(tech)
    os.makedirs(out_dir, exist_ok=True)
    best_path = os.path.join(out_dir, "lsi_best.pt")
    pred_path = os.path.join(out_dir, "prediction.json")
    summ_path = os.path.join(out_dir, "summary.json")
    prob_path = os.path.join(out_dir, "doc_probs.pkl")
    if SKIP_DONE and all(os.path.exists(p) for p in [pred_path, summ_path, prob_path]):
        print(f"\n##### [{tech}/{DOC_MODE}] already finished -> skipping")
        return json.load(open(summ_path))

    print(f"\n{'='*78}\n##### Phase 5 | Arch1 | {tech} | doc mode {DOC_MODE} | pool {POOL} | seed {SEED} | {HP[tech]}\n{'='*78}")
    set_seed(SEED)
    if torch.cuda.is_available():
        torch.cuda.reset_peak_memory_stats()
    t_run = time.time()
    loader = make_loader(fit_docs, True)
    teacher = None
    if tech == "kd":
        teacher = build_model(TEACHER_TECH, keep_layers=TEACHER_KEEP, pool_tok=TEACHER_POOL_TOK)
        t_ckpt = TEACHER_CKPT or os.path.join(out_dir_for(TEACHER_TECH), "lsi_best.pt")
        if os.path.exists(t_ckpt):
            teacher.load_state_dict(torch.load(t_ckpt, map_location="cpu")["model"])
            print("Loaded teacher from", t_ckpt)
        else:
            print(f"No saved teacher found -> training the teacher ({TEACHER_TECH}) first")
            fit(teacher, loader, best_path=os.path.join(out_dir, "teacher_best.pt"), **HP[TEACHER_TECH])
        for p in teacher.parameters():
            p.requires_grad = False
        teacher.eval()
        if eval_docs_epoch:
            Ye = label_matrix(eval_docs_epoch); Pe = prob_matrix(predict_probs(teacher, eval_docs_epoch), eval_docs_epoch)
            print(f"TEACHER dev(sub) macro-F1: {best_global_thr(Ye, Pe)[0]:.4f}")
        model = build_model("full", keep_layers=STUDENT_KEEP)
        t_p, s_p = sum(p.numel() for p in teacher.parameters()), sum(p.numel() for p in model.parameters())
        print(f"Teacher {t_p/1e6:.1f}M -> student {s_p/1e6:.1f}M params ({100*(1-s_p/t_p):.0f}% smaller)")
        history = fit(model, loader, best_path=best_path, teacher=teacher, **HP["kd"])
    else:
        model = build_model(tech)
        history = fit(model, loader, best_path=best_path, **HP[tech])
    json.dump(history, open(os.path.join(out_dir, "history.json"), "w"), indent=2)
    train_min = (time.time() - t_run) / 60
    print(f"[{tech}] training finished in {train_min:.1f} min")

    dev_f1, thr, dev_probs = None, THR_FINAL, {}
    if not FINAL_ALL:
        dev_probs = predict_probs(model, dev_docs)
        dev_f1, thr = report(dev_probs, dev_docs)
    t_inf = time.time()
    test_probs = predict_probs(model, test_docs)
    inf_sec = time.time() - t_inf
    pred = make_submission(prob_matrix(test_probs, test_docs), thr)
    assert len(pred) == len(test_docs) and {p["id"] for p in pred} == {d["id"] for d in test_docs}
    print(f"[{tech}] test docs {len(pred)} | avg #statutes per doc {np.mean([len(p['pred']) for p in pred]):.2f} "
          f"| threshold {thr:.2f} | test inference {inf_sec:.0f}s")
    json.dump(pred, open(pred_path, "w"))
    pickle.dump({"all_labels": all_labels, "dev": {k: v.astype(np.float16) for k, v in dev_probs.items()},
                 "test": {k: v.astype(np.float16) for k, v in test_probs.items()}}, open(prob_path, "wb"))
    n_tot = sum(p.numel() for p in model.parameters())
    n_trn = sum(p.numel() for p in model.parameters() if p.requires_grad)
    summary = {"technique": tech, "doc_mode": DOC_MODE, "pool": POOL, "dev_f1": dev_f1, "thr": thr,
               "train_minutes": round(train_min, 1), "test_inference_sec": round(inf_sec, 1),
               "params_total_M": round(n_tot / 1e6, 2), "params_trainable_M": round(n_trn / 1e6, 3),
               "peak_gpu_mem_GB": round(torch.cuda.max_memory_allocated() / 1e9, 2) if torch.cuda.is_available() else None}
    json.dump(summary, open(summ_path, "w"), indent=2)
    del model, teacher
    return summary


# %% [10] Run everything chosen in cell [2]
all_results = []
for _mode in DOC_MODES:
    DOC_MODE = _mode
    for tech in TECHNIQUES:
        try:
            res = run_technique(tech)
        except Exception:
            print(f"\n!!!!! [{tech}/{DOC_MODE}] FAILED -- continuing")
            traceback.print_exc()
            res = {"technique": tech, "doc_mode": DOC_MODE, "error": True}
        all_results.append(res)
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

print("\n" + "=" * 78 + "\nSUMMARY (dev macro-F1 on the full dev set, global threshold tuned on dev)\n" + "=" * 78)
for r in all_results:
    if r.get("error"):
        print(f"{r['technique']:9s} {r['doc_mode']:9s} FAILED")
    else:
        d = f"{r['dev_f1']:.4f}" if r["dev_f1"] is not None else "n/a"
        print(f"{r['technique']:9s} {r['doc_mode']:9s} dev macro-F1 {d} thr {r['thr']:.2f} | train {r['train_minutes']} min | "
              f"params {r['params_total_M']}M (trainable {r['params_trainable_M']}M) | mem {r['peak_gpu_mem_GB']} GB")


# %% [11] Zips: one Codabench submission zip per run (prediction.json at the zip root) + a results zip
SUB_DIR, RES_DIR = f"{ROOT_OUT}/submissions", f"{ROOT_OUT}/results_zips"
os.makedirs(SUB_DIR, exist_ok=True); os.makedirs(RES_DIR, exist_ok=True)
for _mode in DOC_MODES:
    DOC_MODE = _mode
    for t in TECHNIQUES:
        d = out_dir_for(t)
        if not os.path.exists(f"{d}/prediction.json"):
            print(f"[{t}/{_mode}] prediction.json not found -> skipped"); continue
        tag = f"phase5_arch1_{t}_{_mode}_{POOL}" + ("" if POOL_TOK == "mean" else "_" + POOL_TOK) + f"_s{SEED}" + ("_all" if FINAL_ALL else "")
        with zipfile.ZipFile(f"{SUB_DIR}/submission_{tag}.zip", "w", zipfile.ZIP_DEFLATED) as zf:
            zf.write(f"{d}/prediction.json", arcname="prediction.json")
        with zipfile.ZipFile(f"{RES_DIR}/results_{tag}.zip", "w", zipfile.ZIP_DEFLATED) as zf:
            for fn in ["prediction.json", "doc_probs.pkl", "history.json", "summary.json"]:
                if os.path.exists(f"{d}/{fn}"):
                    zf.write(f"{d}/{fn}", arcname=fn)
        print(f"[{t}/{_mode}] submission_{tag}.zip")
print("Upload the zips in", SUB_DIR, "to Codabench.")


# %% [12] OPTIONAL: ensemble + threshold study on saved runs (CPU only: turn the GPU off). Needs cells [1]-[4] and [7] run first.
# Upload each run's doc_probs.pkl (from the results zips) as a Kaggle dataset and list them here.
ENSEMBLE_MEMBERS = {   # name: (path to doc_probs.pkl, weight)
    # "lorma":   ("/kaggle/input/YOUR_DS/doc_probs_lorma.pkl",   1),
    # "adapter": ("/kaggle/input/YOUR_DS/doc_probs_adapter.pkl", 1),
    # "full":    ("/kaggle/input/YOUR_DS/doc_probs_full.pkl",    1),
    # "kd":      ("/kaggle/input/YOUR_DS/doc_probs_kd.pkl",      1),
}
ENSEMBLE_USE = None       # None = all members, or a list of names
THR_OVERRIDE = None       # force this global threshold for the test mix (REQUIRED if no member has dev probabilities)
PER_LABEL_MIN_POS = 8     # per-label thresholds only for labels with >= this many dev positives
if ENSEMBLE_MEMBERS:
    import itertools
    data = {n: pickle.load(open(p, "rb")) for n, (p, w) in ENSEMBLE_MEMBERS.items()}
    W = {n: w for n, (p, w) in ENSEMBLE_MEMBERS.items()}
    names = list(data)
    dev_names = [n for n in names if data[n]["dev"]]      # FINAL_ALL runs have no dev probabilities: they only join the TEST mix
    assert dev_names or THR_OVERRIDE is not None, "no member has dev probabilities: set THR_OVERRIDE"
    for n in names:
        assert data[n]["all_labels"] == all_labels, f"{n}: label space differs"
        assert set(data[n]["test"]) == {x["uid"] for x in test_docs}, f"{n}: test keys differ"
    def mixP(split, docs, use):
        if split == "dev":
            use = [n for n in use if data[n]["dev"]]
        tot = sum(W[n] for n in use)
        return sum(W[n] * np.stack([data[n][split][d["uid"]].astype(np.float32) for d in docs]) for n in use) / tot
    use = ENSEMBLE_USE or names
    if dev_names:
        Yd = label_matrix(dev_docs)
        for n in dev_names:
            s, t = best_global_thr(Yd, mixP("dev", dev_docs, [n]))
            print(f"member {n:10s} dev macro-F1 {s:.4f} @thr {t:.2f}")
        print("test-only members (no dev probabilities):", [n for n in names if n not in dev_names])
        if len(dev_names) > 2:
            rows = sorted(((best_global_thr(Yd, mixP("dev", dev_docs, list(c)))[0], c) for r in range(1, len(dev_names) + 1)
                           for c in itertools.combinations(dev_names, r)), reverse=True)
            print("best subsets (dev is noisy: differences < ~0.01 mean nothing):")
            for f_, c in rows[:8]:
                print(f"  {f_:.4f}  {c}")
        Pd = mixP("dev", dev_docs, use)
        g_f1, g_thr = best_global_thr(Yd, Pd)
        print(f"selected members {use}: global threshold {g_thr:.2f} -> dev macro-F1 {g_f1:.4f}")

        def fit_perlabel(Y, P, base_thr, min_pos):
            thr = np.full(N_LAB, base_thr)
            for j in range(N_LAB):
                if Y[:, j].sum() >= min_pos:
                    best = max(((f1_per_label(Y[:, [j]], P[:, [j]] >= t)[0], t) for t in THR_GRID))
                    thr[j] = best[1]
            return thr
        rng = np.random.RandomState(0); idx = rng.permutation(len(dev_docs)); A, B = idx[: len(idx) // 2], idx[len(idx) // 2:]
        gains = []
        for fit_i, ev_i in [(A, B), (B, A)]:
            gthr = best_global_thr(Yd[fit_i], Pd[fit_i])[1]
            pthr = fit_perlabel(Yd[fit_i], Pd[fit_i], gthr, PER_LABEL_MIN_POS)
            g = macro_f1(Yd[ev_i], apply_thr(Pd[ev_i], pthr)) - macro_f1(Yd[ev_i], apply_thr(Pd[ev_i], gthr))
            gains.append(g); print(f"per-label thresholds, held-out half gain: {g:+.4f}")
        use_perlabel = float(np.mean(gains)) > 0.005
        print("mean held-out gain", round(float(np.mean(gains)), 4), "-> per-label thresholds:", use_perlabel)
        final_thr = fit_perlabel(Yd, Pd, g_thr, PER_LABEL_MIN_POS) if use_perlabel else g_thr
    else:
        final_thr = float(THR_OVERRIDE)
    if THR_OVERRIDE is not None:
        final_thr = float(THR_OVERRIDE); print('forced global threshold', final_thr)
    Pt = mixP("test", test_docs, use)
    pred = make_submission(Pt, final_thr)
    ED = f"{ROOT_OUT}/phase5_ensemble"; os.makedirs(ED, exist_ok=True)
    json.dump(pred, open(f"{ED}/prediction.json", "w"))
    with zipfile.ZipFile(f"{ED}/submission_phase5_ensemble.zip", "w", zipfile.ZIP_DEFLATED) as zf:
        zf.write(f"{ED}/prediction.json", arcname="prediction.json")
    print("avg #statutes per doc", round(float(np.mean([len(p["pred"]) for p in pred])), 2), "| wrote", f"{ED}/submission_phase5_ensemble.zip")

Phase_5_LSI/Train/train.json: reconstructing file:   0%|          |  0.00B /  299MB            

Phase_5_LSI/Train/train.json: downloading bytes:           |  0.00B            

train: 42750 records | keys of first record: ['id', 'text', 'labels']
  tokenised 2000/42750 docs (4s)
  tokenised 4000/42750 docs (6s)
  tokenised 6000/42750 docs (9s)
  tokenised 8000/42750 docs (11s)
  tokenised 10000/42750 docs (13s)
  tokenised 12000/42750 docs (15s)
  tokenised 14000/42750 docs (17s)
  tokenised 16000/42750 docs (19s)
  tokenised 18000/42750 docs (21s)
  tokenised 20000/42750 docs (22s)
  tokenised 22000/42750 docs (24s)
  tokenised 24000/42750 docs (26s)
  tokenised 26000/42750 docs (27s)
  tokenised 28000/42750 docs (29s)
  tokenised 30000/42750 docs (31s)
  tokenised 32000/42750 docs (32s)
  tokenised 34000/42750 docs (34s)
  tokenised 36000/42750 docs (36s)
  tokenised 38000/42750 docs (37s)
  tokenised 40000/42750 docs (39s)
  tokenised 42000/42750 docs (41s)
  train: removed 0 exact duplicates | 0 further docs share an id with different content (kept)
  train ready in 41s


Phase_5_LSI/Train/dev.json: reconstructing file:   0%|          |  0.00B / 79.4MB            

Phase_5_LSI/Train/dev.json: downloading bytes:           |  0.00B            

dev: 10181 records | keys of first record: ['id', 'text', 'labels']
  tokenised 2000/10181 docs (2s)
  tokenised 4000/10181 docs (4s)
  tokenised 6000/10181 docs (6s)
  tokenised 8000/10181 docs (8s)
  tokenised 10000/10181 docs (10s)
  dev: removed 0 exact duplicates | 0 further docs share an id with different content (kept)
  dev ready in 11s


Phase_5_LSI/Test/test.json: reconstructing file:   0%|          |  0.00B / 98.1MB            

Phase_5_LSI/Test/test.json: downloading bytes:           |  0.00B            

test: 13019 records | keys of first record: ['id', 'text']
  tokenised 2000/13019 docs (2s)
  tokenised 4000/13019 docs (4s)
  tokenised 6000/13019 docs (6s)
  tokenised 8000/13019 docs (8s)
  tokenised 10000/13019 docs (10s)
  tokenised 12000/13019 docs (11s)
  test ready in 12s
label space: 100 statute ids (min id 0, max id 99)
train: 42750 docs | tokens median 630 p90 4085 max 33489 | chunks(1023) median 1 p90 4
   labels per doc: mean 3.70 median 3 max 31
dev: 10181 docs | tokens median 788 p90 4417 max 30018 | chunks(1023) median 1 p90 5
   labels per doc: mean 3.92 median 3 max 28
test: 13019 docs | tokens median 780 p90 4229 max 33498 | chunks(1023) median 1 p90 5
train positives per label: median 679 | max 11730 | labels with <5: 0 | <10: 0 | <20: 0
most frequent statute ids (id: train count): {5: 11730, 34: 10380, 97: 7880, 43: 7728, 39: 7120, 12: 4324, 13: 4136, 10: 4134, 77: 4046, 94: 3835}
dev: 100 labels have >=1 dev positive | 0 of them never appear in train
fit docs 5293